# Simulations paramétriques de Pouls

Ce notebook reprend les règles du moteur de `model.js` avec un électorat **uniforme** et **sans l'émergence ni le remplacement de candidats**. Le nombre de candidats et leurs places restent fixes pendant toute une simulation.

Pour chaque combinaison de paramètres, il lance `N_REPLICATES` simulations indépendantes de `T_PERIODS` périodes. Chaque réplication a une graine reproductible ; les mêmes graines sont réutilisées d'une combinaison à l'autre afin de faciliter leur comparaison. Le vote initial est calculé à `t = 0` et sert de première observation pour la réputation.

Modifiez `BASE_PARAMETERS` pour changer les valeurs de référence, puis `PARAMETER_GRID` pour indiquer les paramètres à balayer et leurs valeurs. Les paramètres absents de la grille conservent leur valeur de référence. Une grille vide correspond à une seule combinaison de référence.

Le calcul utilise NumPy, pandas et Matplotlib. Les valeurs par défaut de résolution (1 600 points d'électorat et 200 pas de recherche) correspondent au moteur actuel.

In [ ]:
from __future__ import annotations

from datetime import datetime
from hashlib import sha256
from itertools import product
import math
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Valeurs initiales de l'application.
BASE_PARAMETERS = {
    "alpha": 1.0,
    "beta": 3.5,
    "gamma": 0.1,
    "eta": 0.04,
    "k0": 1.0,
    "k_max": 1.25,
    "noise_sigma": 0.1,
}

# Exemple : PARAMETER_GRID = {"alpha": [0.5, 1.0, 1.5, 2.0], "eta": [0.0, 0.04]}
# Les listes peuvent aussi être créées avec inclusive_values ci-dessous.
PARAMETER_GRID = {}

N_REPLICATES = 3
MASTER_SEED = 314159
T_PERIODS = 500
INTEGRATION_POINTS = 1600
SEARCH_STEPS = 200
INITIAL_POSITIONS = [0.0, 0.0, 0.0, 0.0]
EXPORT_CSV = True
DATA_DIR = Path("simulation") / "data"
RUN_ID = None

## Règles reprises

- Électorat uniforme sur `[-1, 1]`, discrétisé aux points médians et normalisé à une masse totale de 1.
- Coût candidat-électeur `max(0, k_i + beta * |x - p_i|**alpha + epsilon)`. Le seuil `k_max` exclut les candidats trop coûteux ; les candidats admissibles se partagent les voix selon le softmax de température 1. Sans candidat admissible, l'électeur s'abstient.
- Bruit gaussien additif indépendant par électeur et candidat, fixé pour toutes les positions testées pendant une même période, puis renouvelé à la période suivante.
- Chaque candidat choisit sa meilleure réponse contre les positions et coûts de début de période. Les décisions sont ensuite appliquées simultanément. La recherche couvre la grille de pas `2 / SEARCH_STEPS` dans `[p_i - gamma, p_i + gamma]`, bornée à `[-1, 1]`, ainsi que la position actuelle et les bornes accessibles. La position actuelle est conservée en cas d'absence de gain strict.
- Les coûts évoluent avant les décisions : `k_i <- k_i * exp(eta * (1/N - S_i))`. `S_i` est la moyenne pondérée des trois dernières parts des exprimés (poids 3, 2, 1, de la plus récente à la plus ancienne).
- Le vote initial à `t = 0` initialise l'historique des parts. Les candidats, leur nombre et leurs identités ne changent jamais ; il n'y a pas de mécanisme d'entrée, de remplacement ni de réinitialisation de réputation.

La grille conserve l'ordre des candidats pour les calculs, mais chaque meilleure réponse est calculée à partir du même état de début de période : l'ordre n'influence donc pas le résultat.

In [ ]:
EPS = 1e-10
GRID_PARAMETERS = {"alpha", "beta", "gamma", "eta", "k0", "k_max", "noise_sigma"}


def inclusive_values(start: float, stop: float, count: int) -> list[float]:
    """Construit count valeurs régulièrement espacées, bornes incluses."""
    if not isinstance(count, int) or count < 1:
        raise ValueError("count doit être un entier positif.")
    if count == 1:
        return [float(start)]
    return [float(start + (stop - start) * i / (count - 1)) for i in range(count)]


def validate(parameters: dict, positions: np.ndarray) -> None:
    numeric = [
        parameters[key] for key in GRID_PARAMETERS - {"k_max"}
    ]
    if not all(math.isfinite(value) for value in numeric):
        raise ValueError("Tous les paramètres numériques doivent être finis.")
    if parameters["k_max"] is not None and (
        not math.isfinite(parameters["k_max"]) or parameters["k_max"] < 0
    ):
        raise ValueError("k_max doit être positif ou nul, ou None pour désactiver le seuil.")
    if parameters["alpha"] <= 0 or parameters["beta"] < 0 or parameters["eta"] < 0:
        raise ValueError("alpha doit être positif ; beta et eta doivent être positifs ou nuls.")
    if not 0 <= parameters["gamma"] <= 2:
        raise ValueError("gamma doit appartenir à [0, 2].")
    if parameters["k0"] <= 0 or parameters["noise_sigma"] < 0:
        raise ValueError("k0 doit être positif et noise_sigma positif ou nul.")
    if positions.ndim != 1 or not 2 <= len(positions) <= 7:
        raise ValueError("Il faut entre 2 et 7 positions initiales.")
    if not np.all(np.isfinite(positions)) or np.any(positions < -1) or np.any(positions > 1):
        raise ValueError("Les positions doivent être finies et appartenir à [-1, 1].")


def build_electorate(parameters: dict, count: int) -> tuple[np.ndarray, np.ndarray]:
    if not isinstance(count, int) or count < 2:
        raise ValueError("INTEGRATION_POINTS doit être un entier supérieur ou égal à 2.")
    dx = 2.0 / count
    x = -1.0 + (np.arange(count, dtype=float) + 0.5) * dx
    mass = np.full(count, 1.0 / count)
    return x, mass


def seed_for(run_seed: int, period: int) -> int:
    text = f"pouls:{run_seed}:{period}".encode("utf-8")
    return int.from_bytes(sha256(text).digest()[:4], "little")


def noise_shocks(point_count: int, candidate_count: int, sigma: float, seed: int) -> np.ndarray:
    """Reproduit le générateur LCG et Box-Muller de model.js."""
    if not isinstance(seed, int) or not 0 <= seed <= 0xFFFFFFFF:
        raise ValueError("La graine doit être un entier non signé de 32 bits.")
    state = seed

    def random() -> float:
        nonlocal state
        state = (1664525 * state + 1013904223) & 0xFFFFFFFF
        return state / 0x100000000

    shocks = np.empty((point_count, candidate_count), dtype=float)
    for point in range(point_count):
        for candidate in range(candidate_count):
            gaussian = math.sqrt(-2.0 * math.log(1.0 - random())) * math.cos(2.0 * math.pi * random())
            shocks[point, candidate] = sigma * gaussian
    return shocks


def evaluate(
    x: np.ndarray,
    mass: np.ndarray,
    positions: np.ndarray,
    scales: np.ndarray,
    parameters: dict,
    shocks: np.ndarray,
) -> tuple[np.ndarray, float, np.ndarray]:
    costs = np.maximum(
        0.0,
        scales[None, :] + parameters["beta"] * np.abs(x[:, None] - positions[None, :]) ** parameters["alpha"] + shocks,
    )
    if not np.all(np.isfinite(costs)):
        raise ValueError("Les coûts spatiaux dépassent les limites numériques.")
    eligible = np.ones_like(costs, dtype=bool) if parameters["k_max"] is None else costs <= parameters["k_max"]
    masked = np.where(eligible, costs, np.inf)
    minimum = np.min(masked, axis=1)
    has_eligible = np.isfinite(minimum)
    baseline = np.where(has_eligible, minimum, 0.0)
    shifted = np.where(eligible, costs - baseline[:, None], 0.0)
    weights = np.where(eligible, np.exp(-shifted), 0.0)
    totals = weights.sum(axis=1)
    probabilities = np.divide(
        weights,
        totals[:, None],
        out=np.zeros_like(weights),
        where=totals[:, None] > 0,
    )
    votes = (probabilities * mass[:, None]).sum(axis=0)
    return votes, float(votes.sum()), probabilities


def expressed_shares(votes: np.ndarray, turnout: float) -> np.ndarray:
    return votes / turnout if turnout > EPS else np.zeros_like(votes)


def updated_scales(scales: np.ndarray, history: list[np.ndarray], eta: float) -> np.ndarray:
    count = len(scales)
    recent = history[-3:]
    if not recent:
        success = np.full(count, 1.0 / count)
    else:
        weights = np.array([3.0, 2.0, 1.0][: len(recent)])
        ordered = recent[::-1]
        success = sum(weight * shares for weight, shares in zip(weights, ordered)) / weights.sum()
    result = scales * np.exp(eta * (1.0 / count - success))
    if not np.all(np.isfinite(result)) or np.any(result <= 0):
        raise ValueError("Les coûts liés au succès dépassent les limites numériques.")
    return result


def best_response(
    x: np.ndarray,
    mass: np.ndarray,
    positions: np.ndarray,
    scales: np.ndarray,
    parameters: dict,
    shocks: np.ndarray,
    candidate: int,
    search_steps: int,
) -> float:
    before = evaluate(x, mass, positions, scales, parameters, shocks)[0][candidate]
    best_votes = float(before)
    best_position = float(positions[candidate])
    lower = max(-1.0, positions[candidate] - parameters["gamma"])
    upper = min(1.0, positions[candidate] + parameters["gamma"])
    candidates = {float(positions[candidate]), float(lower), float(upper)}
    candidates.update(
        position
        for j in range(search_steps + 1)
        if lower <= (position := -1.0 + 2.0 * j / search_steps) <= upper
    )

    rivals = [index for index in range(len(positions)) if index != candidate]
    rival_costs = np.maximum(
        0.0,
        scales[rivals][None, :]
        + parameters["beta"] * np.abs(x[:, None] - positions[rivals][None, :]) ** parameters["alpha"]
        + shocks[:, rivals],
    )
    if parameters["k_max"] is None:
        rival_eligible = np.ones_like(rival_costs, dtype=bool)
    else:
        rival_eligible = rival_costs <= parameters["k_max"]
    rival_minimum = np.min(np.where(rival_eligible, rival_costs, np.inf), axis=1)
    has_rival = np.isfinite(rival_minimum)
    rival_baseline = np.where(has_rival, rival_minimum, 0.0)
    rival_shifted = np.where(rival_eligible, rival_costs - rival_baseline[:, None], 0.0)
    rival_weight = np.where(rival_eligible, np.exp(-rival_shifted), 0.0).sum(axis=1)

    search_positions = np.array(sorted(candidates), dtype=float)
    candidate_costs = np.maximum(
        0.0,
        scales[candidate]
        + parameters["beta"] * np.abs(x[:, None] - search_positions[None, :]) ** parameters["alpha"]
        + shocks[:, candidate, None],
    )
    if parameters["k_max"] is None:
        candidate_eligible = np.ones_like(candidate_costs, dtype=bool)
    else:
        candidate_eligible = candidate_costs <= parameters["k_max"]
    difference = np.clip(candidate_costs - rival_minimum[:, None], -745.0, 700.0)
    denominator = 1.0 + rival_weight[:, None] * np.exp(difference)
    scores = np.where(candidate_eligible, mass[:, None] / denominator, 0.0).sum(axis=0)

    for position, score in zip(search_positions, scores):
        if position == positions[candidate]:
            continue
        if score > best_votes + EPS:
            best_votes = float(score)
            best_position = float(position)
    return best_position


def run_simulation(
    parameters: dict,
    run_seed: int,
    initial_positions: list[float],
    periods: int,
    integration_points: int = INTEGRATION_POINTS,
    search_steps: int = SEARCH_STEPS,
) -> tuple[list[dict], dict]:
    positions = np.asarray(initial_positions, dtype=float)
    validate(parameters, positions)
    if not isinstance(periods, int) or periods < 0:
        raise ValueError("periods doit être un entier positif ou nul.")
    if not isinstance(search_steps, int) or search_steps < 2:
        raise ValueError("search_steps doit être un entier supérieur ou égal à 2.")
    x, mass = build_electorate(parameters, integration_points)
    scales = np.full(len(positions), parameters["k0"], dtype=float)
    shocks = noise_shocks(len(x), len(positions), parameters["noise_sigma"], seed_for(run_seed, 0))
    votes, turnout, _ = evaluate(x, mass, positions, scales, parameters, shocks)
    history = [expressed_shares(votes, turnout)]
    trajectory = []

    def record(period: int, period_votes: np.ndarray, period_turnout: float) -> None:
        shares = expressed_shares(period_votes, period_turnout)
        for candidate in range(len(positions)):
            trajectory.append({
                "period": period,
                "candidate": candidate,
                "candidate_label": chr(ord("A") + candidate),
                "position": float(positions[candidate]),
                "cost_scale": float(scales[candidate]),
                "votes_of_electorate": float(period_votes[candidate]),
                "share_of_expressed": float(shares[candidate]),
                "turnout": float(period_turnout),
                "abstention": float(1.0 - period_turnout),
            })

    record(0, votes, turnout)
    for period in range(1, periods + 1):
        scales = updated_scales(scales, history, parameters["eta"])
        shocks = noise_shocks(len(x), len(positions), parameters["noise_sigma"], seed_for(run_seed, period))
        # Toutes les réponses utilisent les mêmes positions et coûts de début de période.
        next_positions = np.array([
            best_response(x, mass, positions, scales, parameters, shocks, candidate, search_steps)
            for candidate in range(len(positions))
        ])
        if np.any(np.abs(next_positions - positions) > parameters["gamma"] + 1e-12):
            raise RuntimeError("Une décision dépasse la borne de déplacement gamma.")
        positions = next_positions
        votes, turnout, _ = evaluate(x, mass, positions, scales, parameters, shocks)
        history.append(expressed_shares(votes, turnout))
        history = history[-3:]
        record(period, votes, turnout)

    recent_abstention = [row["abstention"] for row in trajectory if row["candidate"] == 0 and row["period"] > 0]
    recent_abstention = recent_abstention[-min(100, periods):] if periods else []
    final_rows = [row for row in trajectory if row["period"] == periods]
    summary = {
        "final_turnout": float(final_rows[0]["turnout"]),
        "final_abstention": float(final_rows[0]["abstention"]),
        "mean_last_100_abstention": float(np.mean(recent_abstention)) if recent_abstention else math.nan,
    }
    for row in final_rows:
        label = row["candidate_label"]
        summary[f"final_position_{label}"] = row["position"]
        summary[f"final_cost_{label}"] = row["cost_scale"]
        summary[f"final_vote_{label}"] = row["votes_of_electorate"]
    return trajectory, summary


def expand_parameter_grid(base: dict, grid: dict) -> list[dict]:
    unknown = set(grid) - GRID_PARAMETERS
    if unknown:
        raise ValueError(f"Paramètres inconnus dans PARAMETER_GRID : {sorted(unknown)}")
    keys = list(grid)
    values = [list(grid[key]) for key in keys]
    if any(not choices for choices in values):
        raise ValueError("Chaque paramètre de PARAMETER_GRID doit avoir au moins une valeur.")
    combinations = []
    for combination in product(*values):
        parameters = {**base, **dict(zip(keys, combination))}
        validate(parameters, np.asarray(INITIAL_POSITIONS, dtype=float))
        combinations.append(parameters)
    return combinations or [base.copy()]


def run_parameter_sweep() -> tuple[pd.DataFrame, pd.DataFrame]:
    if not isinstance(N_REPLICATES, int) or N_REPLICATES < 1:
        raise ValueError("N_REPLICATES doit être un entier positif.")
    combinations = expand_parameter_grid(BASE_PARAMETERS, PARAMETER_GRID)
    print(
        f"{len(combinations)} combinaison(s) × {N_REPLICATES} réplication(s) × "
        f"{T_PERIODS} période(s) = {len(combinations) * N_REPLICATES * T_PERIODS:,} périodes simulées."
    )
    summary_rows = []
    trajectory_rows = []
    varied_keys = list(PARAMETER_GRID)
    for scenario_number, parameters in enumerate(combinations, start=1):
        scenario_id = f"S{scenario_number:03d}"
        for replicate in range(N_REPLICATES):
            run_seed = (MASTER_SEED + replicate) & 0xFFFFFFFF
            trajectory, summary = run_simulation(
                parameters, run_seed, INITIAL_POSITIONS, T_PERIODS,
                INTEGRATION_POINTS, SEARCH_STEPS,
            )
            for row in trajectory:
                trajectory_rows.append({
                    "scenario_id": scenario_id,
                    "replicate": replicate + 1,
                    "run_seed": run_seed,
                    **{key: parameters[key] for key in varied_keys},
                    **row,
                })
            summary_rows.append({
                "scenario_id": scenario_id,
                "replicate": replicate + 1,
                "run_seed": run_seed,
                **{key: parameters[key] for key in varied_keys},
                **summary,
            })
    return pd.DataFrame(summary_rows), pd.DataFrame(trajectory_rows)


summary_df, trajectories_df = run_parameter_sweep()
display(summary_df.head())
display(trajectories_df.head())

In [ ]:
# Évolution des parts de marché parmi les exprimés, moyennée sur les répétitions.
# La bande représente moyenne ± 1 écart-type ; variance et écart-type sont calculés à chaque période.
market_stats = (
    trajectories_df.groupby(["scenario_id", "period", "candidate", "candidate_label"])
    .agg(
        mean_market_share=("share_of_expressed", "mean"),
        variance_market_share=("share_of_expressed", "var"),
        sd_market_share=("share_of_expressed", "std"),
    )
    .reset_index()
)
# Avec une seule répétition, la variance d'échantillon est indéfinie : la bande est réduite à la moyenne.
market_stats["sd_market_share"] = market_stats["sd_market_share"].fillna(0.0)
display(market_stats.head(12))

# None trace tous les scénarios ; par exemple ["S001"] limite l'affichage à un seul.
SCENARIOS_TO_PLOT = None
scenario_ids = sorted(market_stats["scenario_id"].unique())
if SCENARIOS_TO_PLOT is not None:
    unknown_scenarios = set(SCENARIOS_TO_PLOT) - set(scenario_ids)
    if unknown_scenarios:
        raise ValueError(f"Scénarios inconnus : {sorted(unknown_scenarios)}")
    scenario_ids = SCENARIOS_TO_PLOT

for scenario_id in scenario_ids:
    scenario_stats = market_stats[market_stats["scenario_id"] == scenario_id]
    parameters = summary_df.loc[summary_df["scenario_id"] == scenario_id, list(PARAMETER_GRID)].iloc[0].to_dict()
    title_parameters = ", ".join(f"{key}={value}" for key, value in parameters.items())
    fig, ax = plt.subplots(figsize=(10, 5))
    for candidate_label, candidate_stats in scenario_stats.groupby("candidate_label", sort=True):
        candidate_stats = candidate_stats.sort_values("period")
        periods = candidate_stats["period"].to_numpy()
        mean = 100.0 * candidate_stats["mean_market_share"].to_numpy()
        sd = 100.0 * candidate_stats["sd_market_share"].to_numpy()
        ax.plot(periods, mean, label=candidate_label)
        ax.fill_between(periods, np.maximum(0.0, mean - sd), np.minimum(100.0, mean + sd), alpha=0.18)
    ax.set_xlabel("Période t")
    ax.set_ylabel("Part de marché parmi les exprimés (%)")
    ax.set_ylim(0, 100)
    ax.set_title(f"{scenario_id} — moyenne ± 1 écart-type" + (f" ({title_parameters})" if title_parameters else ""))
    ax.grid(alpha=0.3)
    ax.legend(title="Candidat")
    plt.show()

In [ ]:
# Résumé moyen par combinaison, puis comparaison graphique.
scenario_columns = ["scenario_id", *PARAMETER_GRID.keys()]
aggregate_df = (
    summary_df.groupby(scenario_columns, dropna=False)
    .agg(
        replicates=("replicate", "count"),
        mean_final_abstention=("final_abstention", "mean"),
        sd_final_abstention=("final_abstention", "std"),
        mean_last_100_abstention=("mean_last_100_abstention", "mean"),
        sd_last_100_abstention=("mean_last_100_abstention", "std"),
    )
    .reset_index()
)
display(aggregate_df)

fig, ax = plt.subplots(figsize=(max(8, len(aggregate_df) * 0.7), 5))
x = np.arange(len(aggregate_df))
error = aggregate_df["sd_last_100_abstention"].fillna(0.0)
ax.errorbar(
    x,
    100 * aggregate_df["mean_last_100_abstention"],
    yerr=100 * error,
    fmt="o",
    capsize=4,
)
ax.set_xticks(x, aggregate_df["scenario_id"])
ax.set_xlabel("Combinaison de paramètres")
ax.set_ylabel("Abstention moyenne sur les dernières périodes (%)")
ax.set_title("Comparaison des scénarios (moyenne ± écart-type entre réplications)")
ax.grid(axis="y", alpha=0.3)
plt.show()

if EXPORT_CSV:
    RUN_ID = datetime.now().strftime("%Y%m%d-%H%M%S-%f")
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    file_prefix = f"simulation-parametrique-{RUN_ID}"
    summary_path = DATA_DIR / f"{file_prefix}-resume.csv"
    trajectories_path = DATA_DIR / f"{file_prefix}-trajectoires.csv"
    metadata_path = DATA_DIR / f"{file_prefix}-parametres.json"
    summary_df.to_csv(summary_path, index=False)
    trajectories_df.to_csv(trajectories_path, index=False)

    def json_compatible(value):
        if isinstance(value, np.generic):
            return value.item()
        if isinstance(value, dict):
            return {str(key): json_compatible(item) for key, item in value.items()}
        if isinstance(value, (list, tuple)):
            return [json_compatible(item) for item in value]
        return value

    metadata = {
        "run_id": RUN_ID,
        "electorate": "uniform",
        "actor_entry": False,
        "base_parameters": BASE_PARAMETERS,
        "parameter_grid": PARAMETER_GRID,
        "replicates": N_REPLICATES,
        "master_seed": MASTER_SEED,
        "periods": T_PERIODS,
        "integration_points": INTEGRATION_POINTS,
        "search_steps": SEARCH_STEPS,
        "initial_positions": INITIAL_POSITIONS,
    }
    with metadata_path.open("w", encoding="utf-8") as metadata_file:
        json.dump(json_compatible(metadata), metadata_file, ensure_ascii=False, indent=2)
    print(f"Jeux de données enregistrés dans {DATA_DIR.resolve()} :")
    print(f"- {summary_path.name}")
    print(f"- {trajectories_path.name}")
    print(f"- {metadata_path.name}")